# 🏎️ ML Racing Simulator — NEAT Trainer (Google Colab)

This notebook trains an AI to race using **NEAT neuroevolution** entirely inside Google Colab.

Because Colab has no display, pygame is initialised with a **dummy video driver** so all the
physics / track geometry still works — we just skip the GUI and plot training progress with
matplotlib instead.

**Workflow:**
1. Clone the repo & install deps  
2. Configure your run (track, population size, generations)  
3. Run the headless training loop  
4. Plot the fitness curve  
5. Export / import the trained population CSV  

## Step 1 — Clone the repository & install dependencies

In [ ]:
# Clone the repository (skip if you already have the files)
!git clone https://github.com/Merlin2LmmL/ML-Racing-Simulator.git
%cd ML-Racing-Simulator

In [ ]:
# Install / upgrade required packages
# torch is optional — the sim falls back to numpy automatically if not found
!pip install -q pygame numpy
# Uncomment the next line for GPU-accelerated neural-network forward passes:
# !pip install -q torch

## Step 2 — Initialise pygame in headless mode

Setting `SDL_VIDEODRIVER=dummy` before importing pygame tells SDL to use a
software-only renderer so `pygame.Surface`, `pygame.draw`, etc. all work without
a real screen — which is exactly what Colab (and any headless server) provides.

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"   # headless — no display needed
os.environ["SDL_AUDIODRIVER"] = "dummy"   # silence audio warnings

import pygame
pygame.init()
# A minimal display surface is required so Track._render() can create Surfaces
pygame.display.set_mode((1, 1))
print("pygame", pygame.version.ver, "— headless mode active")

## Step 3 — Import the simulator modules

In [ ]:
import sys, math, random, csv, copy
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# The repo must be on the Python path
if "." not in sys.path:
    sys.path.insert(0, ".")

# Import shared simulator code
from drift_racing import (
    TRACKS, Track, Car, CarState,
    WIDTH, HEIGHT, FPS,
    normalize, lerp, clamp,
)

# Import NEAT components from "ml training.py"
# The file name has a space, so we use importlib
import importlib.util, pathlib
_spec = importlib.util.spec_from_file_location(
    "ml_training",
    pathlib.Path("ml training.py")
)
_ml = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ml)

# Pull out the classes / helpers we need
Gene            = _ml.Gene
Genome          = _ml.Genome
NeuralNet       = _ml.NeuralNet
Population      = _ml.Population
Agent           = _ml.Agent
build_checkpoints = _ml.build_checkpoints
N_INPUTS        = _ml.N_INPUTS
N_OUTPUTS       = _ml.N_OUTPUTS
PHYSICS_DT      = _ml.PHYSICS_DT

print("Modules loaded. Available tracks:")
for i, t in enumerate(TRACKS):
    print(f"  [{i}] {t['name']}")

## Step 4 — Configure your training run

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
TRACK_INDEX     = 0     # 0=Oval Circuit, 1=Technical Twisty, 2=High-Speed Sweeper, …
POPULATION_SIZE = 30    # Number of cars per generation
NUM_GENERATIONS = 50    # How many generations to train
PLOT_EVERY      = 5     # Re-draw the fitness graph every N generations
CSV_PATH        = "neat_population.csv"  # Where to save/load the population

print(f"Track      : {TRACKS[TRACK_INDEX]['name']}")
print(f"Population : {POPULATION_SIZE}")
print(f"Generations: {NUM_GENERATIONS}")

## Step 5 — Run the headless NEAT training loop

In [ ]:
def run_headless_training(
    track_index: int = TRACK_INDEX,
    population_size: int = POPULATION_SIZE,
    num_generations: int = NUM_GENERATIONS,
    plot_every: int = PLOT_EVERY,
    existing_pop=None,
):
    """
    Train a NEAT population on the chosen track for `num_generations` generations.
    Returns the trained Population object.
    """
    track       = Track(TRACKS[track_index])
    checkpoints = build_checkpoints(track)

    pop = existing_pop if existing_pop is not None else Population(population_size)

    best_history = list(pop.best_fitness_history)  # carry over previous history if resuming

    for gen_num in range(num_generations):
        # ── Create agents for this generation ────────────────────────────────
        agents = [Agent(g, track, checkpoints) for g in pop.genomes]

        # ── Step all agents until every car is dead ───────────────────────────
        while any(a.alive for a in agents):
            alive = [a for a in agents if a.alive]
            inp   = np.stack([a._build_inputs_np(track) for a in alive])
            for i, agent in enumerate(alive):
                out = agent.net.forward_batch(inp[i : i + 1])[0]
                agent.apply_output(out, track)

        # ── Evolve ────────────────────────────────────────────────────────────
        pop.evolve()
        best_f = pop.best_fitness_history[-1]
        best_history.append(best_f)

        abs_gen = pop.generation  # generation counter inside the population
        print(
            f"Gen {abs_gen:>4d} | "
            f"best fitness: {best_f:+8.1f} | "
            f"all-time: {pop.best_ever.fitness:+8.1f}"
        )

        # ── Live fitness plot ─────────────────────────────────────────────────
        if plot_every > 0 and (gen_num + 1) % plot_every == 0:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(best_history, color="orange", linewidth=2)
            ax.set_title(
                f"NEAT Training — {TRACKS[track_index]['name']} "
                f"(gen {abs_gen}, best ever: {pop.best_ever.fitness:+.1f})"
            )
            ax.set_xlabel("Generation")
            ax.set_ylabel("Best fitness")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    return pop


# ── Start training ────────────────────────────────────────────────────────────
trained_pop = run_headless_training()

## Step 6 — Plot final fitness curve

In [ ]:
history = trained_pop.best_fitness_history

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history, color="orange", linewidth=2, label="Best fitness per gen")
ax.fill_between(range(len(history)), history, alpha=0.15, color="orange")
ax.axhline(trained_pop.best_ever.fitness, color="red", linestyle="--",
           label=f"All-time best: {trained_pop.best_ever.fitness:+.1f}")
ax.set_title(f"NEAT Fitness History — {TRACKS[TRACK_INDEX]['name']}")
ax.set_xlabel("Generation")
ax.set_ylabel("Best fitness")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal generations trained : {trained_pop.generation}")
print(f"All-time best fitness     : {trained_pop.best_ever.fitness:+.2f}")
print(f"Best genome hidden nodes  : {trained_pop.best_ever.n_hidden}")
print(f"Best genome connections   : {sum(1 for g in trained_pop.best_ever.genes if g.enabled)}")

## Step 7 — Export the trained population to CSV

In [ ]:
with open(CSV_PATH, "w", newline="") as f:
    csv.writer(f).writerows(trained_pop.to_csv_rows())

print(f"Saved {len(trained_pop.genomes)} genomes → {CSV_PATH}")

# Download the file in Colab
try:
    from google.colab import files
    files.download(CSV_PATH)
except ImportError:
    print("(Not running in Colab — file saved locally.)")

## Step 8 — Import a previously saved population and continue training

Upload your `neat_population.csv` (or use the one saved above) and run the cell below
to resume training from where you left off.

In [ ]:
# ── Upload a CSV in Colab (skip if running locally) ──────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()           # opens a file-picker dialog
    # Assume the uploaded file is the population CSV
    upload_name = list(uploaded.keys())[0]
    with open(CSV_PATH, "wb") as f:
        f.write(uploaded[upload_name])
    print(f"Uploaded and saved as {CSV_PATH}")
except ImportError:
    print("(Not in Colab — using local file.)")

# ── Load the population ───────────────────────────────────────────────────────
with open(CSV_PATH, newline="") as f:
    rows = list(csv.reader(f))

loaded_pop = Population.from_csv_rows(rows)
print(f"Loaded population — generation {loaded_pop.generation}, "
      f"{len(loaded_pop.genomes)} genomes")

# ── Continue training ─────────────────────────────────────────────────────────
EXTRA_GENERATIONS = 20   # how many more generations to run
trained_pop = run_headless_training(
    num_generations=EXTRA_GENERATIONS,
    existing_pop=loaded_pop,
)

## Step 9 — Inspect the best genome's network structure

In [ ]:
IN_LABELS  = ["R-90","R-45","R-20","R0","R+20","R+45","R+90",
              "SPD","CP_ANG","CP_DST","DRFT"]
OUT_LABELS = ["THR","BRK","STR"]

best = trained_pop.best_ever
print(f"Best genome summary")
print(f"  Fitness        : {best.fitness:+.2f}")
print(f"  Hidden nodes   : {best.n_hidden}")
print(f"  Total genes    : {len(best.genes)}")
enabled = [g for g in best.genes if g.enabled]
print(f"  Enabled genes  : {len(enabled)}")
print()
print(f"{'In':>10}  {'Out':>10}  {'Weight':>10}  {'Enabled'}")
print("-" * 48)
for gn in sorted(enabled, key=lambda x: (x.in_node, x.out_node)):
    in_lbl  = IN_LABELS[gn.in_node]  if gn.in_node  < len(IN_LABELS)  else f"H{gn.in_node  - N_INPUTS}"
    out_lbl = OUT_LABELS[gn.out_node - N_INPUTS] if N_INPUTS <= gn.out_node < N_INPUTS + N_OUTPUTS \
              else (f"H{gn.out_node - N_INPUTS}" if gn.out_node >= N_INPUTS else f"I{gn.out_node}")
    print(f"{in_lbl:>10}  {out_lbl:>10}  {gn.weight:>+10.4f}  {'yes' if gn.enabled else 'no'}")